In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, '.')

from Engine.TickData import TickData
from Engine.Backtester import Backtester

In [2]:
# Load ES tick data and aggregate into 250-tick bars
td = TickData('./Data/ESTick.txt', timezone='America/Chicago')
td.aggregate(method='tick', size=1000)


Loading tick data: ./Data/ESTick.txt
Loaded 23,033,489 ticks
Price range: 6751.50 - 7027.25

Aggregated 23,033,489 ticks into 23,033 bars (1000 tick)


TickData(1000 tick bars, n_bars=23,033, from 2026-02-01 23:00:07.009440-06:00 to 2026-03-02 07:28:53.006560-06:00)

In [3]:
# ES contract specs
tick_value = 12.50        # $12.50 per tick (0.25 pts)
contract_multiplier = 50  # $50 per point
initial_equity = 100_000

ohlc = td.get_ohlc()
indices = td.get_indices()
timeframe = td.get_timeframe()
n_bars = len(ohlc)

print(f'Bars: {n_bars:,}')
print(f'Timeframe: {timeframe}')
print(f'Date range: {indices[0]} to {indices[-1]}')

Bars: 23,033
Timeframe: 2min
Date range: 2026-02-01 23:00:07.009440-06:00 to 2026-03-02 07:28:53.006560-06:00


In [4]:
# Look-ahead "cheat" strategy:
# - When flat, peek 20 bars into the future
# - If price will be higher → go long, if lower → go short
# - TP = 10 ticks, SL = 10 ticks (1:1 RR)

tp_ticks = 20
sl_ticks = 40
tp_points = tp_ticks * 0.25  # 2.5 points
sl_points = sl_ticks * 0.25  # 2.5 points

lookahead = 100

signal = np.zeros(n_bars)
position = 0
entry_price = 0.0

for i in range(n_bars):
    close = ohlc[i, 3]
    high = ohlc[i, 1]
    low = ohlc[i, 2]

    if position != 0:
        hit_tp = False
        hit_sl = False

        if position == 1:
            hit_tp = high >= entry_price + tp_points
            hit_sl = low <= entry_price - sl_points
        else:
            hit_tp = low <= entry_price - tp_points
            hit_sl = high >= entry_price + sl_points

        if hit_tp or hit_sl:
            position = 0
    else:
        # Look ahead 20 bars — if we can see the future, use it
        future_idx = i + lookahead
        if future_idx < n_bars:
            future_close = ohlc[future_idx, 3]
            if future_close > close:
                position = 1
            elif future_close < close:
                position = -1
            entry_price = close

    signal[i] = position

print(f'Total entries: {int(np.sum(np.abs(np.diff(signal)) > 0))}')
print(f'Long entries: {int(np.sum(np.diff(signal) > 0))}')
print(f'Short entries: {int(np.sum(np.diff(signal) < 0))}')

Total entries: 3357
Long entries: 1679
Short entries: 1678


In [5]:
# Run backtester — no slippage, $5 round trip commission
bt = Backtester(
    ohlc=ohlc,
    indicies=indices,
    signal=signal,
    timeframe=timeframe,
    tick_value=tick_value,
    contract_multiplier=contract_multiplier,
    initial_equity=initial_equity,
    slippage_ticks=1,
    commission=3.5,
)
bt.full_backtest()
print(bt)


        Backtest Results from 2026-02-01 23:00:07.009440-06:00 to 2026-03-02 07:28:53.006560-06:00 (28 days 08:28:45.997120)
        Equity:     $100,000.00 -> $249,223.50
        Buy & Hold: $100,000.00 -> $92,375.00
        Buy & Hold Return (%): -7.62%
        Strategy Return (%):   149.22%

        Total Trades:     1679.00
        Win Rate:         79.57%
        95% CI:           77.58% - 81.43%
        Avg. Win:         $234.96
        Avg. Loss:        $-480.14
        Ratio Win/Loss:   0.49
        Expected Value:   $88.88

        Largest Win:      $2,484.00
        Largest Loss:     $-941.00
        Max Consec. Wins: 21
        Max Consec. Loss: 4

        Max Drawdown from 2026-02-13 13:37:50.005880-06:00 to 2026-02-13 14:42:34.000360-06:00 (0 days 01:04:43.994480)
        Max Drawdown:     $-2,988.50
        Max Drawdown (%): -1.66%

        Slippage:   1 ticks
        Commission: $3.50
        Contract Multiplier: $50.00
        Transaction Costs:   $26,864.00

        C

In [6]:
from Engine.PropFirm import prop_firms, PropFirmSimulator

sim = PropFirmSimulator(bt, prop_firms[2], num_simulations=10000)
sim.run()
sim.summary()



0.7957117331745086 0.48936503132251963 0.1851052304031484
=== Lucid 50K Flex Challenge Simulation (10,000 runs) ===
Pass rate:          94.43%
Avg days to pass:   2.0
Median days to pass: 2.0
Avg trades/day:     64.58
Trade pool size:    1679
Avg attempts to pass: 1.06
Expected cost to pass: $74.13
Optimal sizing: N/A
Optimal sizing (micros): N/A


In [8]:
from Engine.Report import generate_report
generate_report(bt, 'backtest_report.html', open_browser=True, propfirm=prop_firms[2])

0.7957117331745086 0.48936503132251963 0.1851052304031484


ValueError: Invalid format specifier '.1f if sim.ch_avg_attempts_to_pass else "N/A"' for object of type 'float'